# 10 — Evaluation & Comparison (LLM-as-a-Judge Ground Truth Match)
**Project:** Semantic Book Recommender — IT4142 HUST
**Input:**
- `data/processed/books_with_emotions.csv`
- `data/eval/qrels.json` (TREC-style ground truth with multi-relevance)
- All model and index artifacts (`models/`, `data/chroma_db/`)

**Output:**
- `reports/evaluation_final.json` — Precision@5, Recall@5, MRR, NDCG@10, and MAP metrics
- `reports/figures/eval_precision_bar.png`
- `reports/figures/eval_latency_quality.png`

## 1. Khởi tạo & Định nghĩa Đường dẫn

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import time
import math
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from chromadb.config import Settings

# 1. Paths configuration
DATA_PATH    = Path('data/processed/books_with_emotions.csv')
EVAL_PATH    = Path('data/eval/qrels.json')
MODEL_PATH   = Path('models')
CHROMA_PATH  = Path('data/chroma_db')
REPORT_PATH  = Path('reports')
FIGURE_PATH  = Path('reports/figures')

# Create necessary directories
REPORT_PATH.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

# Matplotlib style setup
plt.rcParams.update({
    'figure.dpi': 150, 
    'savefig.dpi': 150,
    'figure.facecolor': 'white', 
    'axes.facecolor': '#F9F9F9',
    'axes.spines.top': False, 
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 
    'axes.labelsize': 11,
})

MODEL_COLORS = {
    'TF-IDF':   '#6B8CBA',
    'BM25':     '#F4A261',
    'Semantic': '#2A9D8F',
    'Hybrid':   '#E76F51',
    'Reranking':'#8338EC',
}

## 2. Load Dữ liệu & Nhãn Đánh giá (qrels.json)

In [ ]:
print("Loading data and test queries...")
df = pd.read_csv(DATA_PATH)
df['isbn13'] = df['isbn13'].astype(str)
print(f"✓ Books loaded: {len(df):,}")

with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    test_queries = json.load(f)
print(f"✓ Test queries loaded from qrels: {len(test_queries)}")

## 3. Tải toàn bộ Model & Index

In [ ]:
print("Loading TF-IDF...")
with open(MODEL_PATH / 'tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
tfidf_matrix = sp.load_npz(MODEL_PATH / 'tfidf_matrix.npz')
print(f"  TF-IDF matrix shape: {tfidf_matrix.shape}")

print("Loading BM25...")
with open(MODEL_PATH / 'bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)
print(f"  BM25 corpus size: {bm25.corpus_size}")

print("Loading BGE-small (Bi-Encoder)...")
bi_encoder = SentenceTransformer('BAAI/bge-small-en-v1.5')

print("Loading ChromaDB...")
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)
collection = chroma_client.get_collection('books')
print(f"  ChromaDB item count: {collection.count():,}")

print("Loading Cross-Encoder for Reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

print("\nAll models loaded successfully! ✓")

## 4. Định nghĩa các hàm Tìm kiếm (Retrieval Functions)

In [ ]:
BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

# A. TF-IDF Search
def search_tfidf(query: str, top_k: int = 10) -> pd.DataFrame:
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx = np.argsort(scores)[::-1][:top_k]
    res = df.iloc[idx][['isbn13', 'title', 'categories', 'average_rating']].copy()
    res['score'] = scores[idx]
    return res.reset_index(drop=True)

# B. BM25 Search
def search_bm25(query: str, top_k: int = 10) -> pd.DataFrame:
    scores = bm25.get_scores(query.lower().split())
    idx = np.argsort(scores)[::-1][:top_k]
    res = df.iloc[idx][['isbn13', 'title', 'categories', 'average_rating']].copy()
    res['score'] = scores[idx]
    return res.reset_index(drop=True)

# C. Semantic Search (BGE-small)
def search_semantic(query: str, top_k: int = 10) -> pd.DataFrame:
    q_emb = bi_encoder.encode([BGE_PREFIX + query], normalize_embeddings=True)
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['metadatas', 'distances', 'documents']
    )
    rows = []
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        rows.append({
            'isbn13'         : results['ids'][0][i],
            'title'          : meta.get('title', ''),
            'authors'        : meta.get('authors', ''),
            'categories'     : meta.get('categories', ''),
            'average_rating' : meta.get('average_rating', 0.0),
            'description'    : results['documents'][0][i],
            'score'          : round(1 - results['distances'][0][i], 4),
        })
    return pd.DataFrame(rows)

# D. Hybrid Search (RRF: BM25 + BGE-small)
def reciprocal_rank_fusion(ranked_lists, k=60):
    scores = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, doc_id in enumerate(ranked_list):
            scores[doc_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def search_hybrid(query: str, top_k: int = 10, candidate_pool: int = 50) -> pd.DataFrame:
    bm25_res = search_bm25(query, top_k=candidate_pool)
    dense_res = search_semantic(query, top_k=candidate_pool)
    bm25_ids = bm25_res['isbn13'].astype(str).tolist()
    dense_ids = dense_res['isbn13'].astype(str).tolist()
    
    fused = reciprocal_rank_fusion([bm25_ids, dense_ids], k=60)
    top_ids = [doc_id for doc_id, _ in fused[:top_k]]
    top_scores = {doc_id: score for doc_id, score in fused[:top_k]}
    
    subset = df[df['isbn13'].astype(str).isin(top_ids)].copy()
    subset['isbn_str'] = subset['isbn13'].astype(str)
    order = {isbn: i for i, isbn in enumerate(top_ids)}
    subset['_order'] = subset['isbn_str'].map(order)
    result = subset.sort_values('_order').drop(columns=['_order', 'isbn_str']).reset_index(drop=True)
    result['score'] = result['isbn13'].astype(str).map(top_scores).round(6)
    return result

# E. Reranking Search
def search_reranking(query: str, top_k: int = 10, candidate_pool: int = 20) -> pd.DataFrame:
    candidates = search_semantic(query, top_k=candidate_pool)
    if candidates.empty:
        return candidates
    pairs = [(query, desc) for desc in candidates['description']]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    candidates['rerank_score'] = scores
    result = candidates.sort_values('rerank_score', ascending=False).head(top_k)
    result['score'] = result['rerank_score']
    return result.reset_index(drop=True)

## 5. Định nghĩa các Hàm tính Chỉ số Đánh giá IR (Precision, Recall, MRR, NDCG, MAP)

In [ ]:
def precision_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    if k <= 0 or not retrieved: return 0.0
    top_k = retrieved[:k]
    hits = sum(1 for isbn in top_k if isbn in relevant)
    return hits / k

def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    if not relevant or k <= 0: return 0.0
    top_k = retrieved[:k]
    hits = sum(1 for isbn in top_k if isbn in relevant)
    return hits / len(relevant)

def calculate_mrr(retrieved: list[str], relevant: set[str], max_k: int = 10) -> float:
    for rank, isbn in enumerate(retrieved[:max_k], start=1):
        if isbn in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    if not relevant or k <= 0: return 0.0
    
    def _dcg(ranked, rel, depth):
        gain = 0.0
        for i, isbn in enumerate(ranked[:depth], start=1):
            if isbn in rel:
                gain += 1.0 / math.log2(i + 1)
        return gain
        
    dcg = _dcg(retrieved, relevant, k)
    ideal_top = ["rel"] * min(len(relevant), k)
    ideal_rel = {"rel"}
    idcg = _dcg(ideal_top, ideal_rel, k)
    return dcg / idcg if idcg > 0.0 else 0.0

def map_score(retrieved: list[str], relevant: set[str]) -> float:
    if not relevant: return 0.0
    hits = 0
    sum_precision = 0.0
    for rank, isbn in enumerate(retrieved, start=1):
        if isbn in relevant:
            hits += 1
            sum_precision += hits / rank
    return sum_precision / len(relevant) if hits > 0 else 0.0

## 6. Chạy Đánh giá Toàn bộ Hệ thống

In [ ]:
SEARCH_FNS = {
    'TF-IDF': search_tfidf,
    'BM25': search_bm25,
    'Semantic': search_semantic,
    'Hybrid': search_hybrid,
    'Reranking': search_reranking
}

eval_records = []
speed_results = {}

print("Evaluating all models against qrels...")
for model_name, fn in SEARCH_FNS.items():
    print(f"  Evaluating {model_name}...")
    t0 = time.time()
    for q in test_queries:
        t_q = time.time()
        res = fn(q['query'], top_k=10)
        lat = (time.time() - t_q) * 1000  # ms
        
        retrieved = res['isbn13'].astype(str).tolist()
        relevant = set(q['relevant_isbns'])
        
        p5 = precision_at_k(retrieved, relevant, k=5)
        r5 = recall_at_k(retrieved, relevant, k=5)
        mrr_val = calculate_mrr(retrieved, relevant, max_k=10)
        ndcg10 = ndcg_at_k(retrieved, relevant, k=10)
        ap_score = map_score(retrieved, relevant)
        
        eval_records.append({
            'model': model_name,
            'query': q['query'],
            'P@5': p5,
            'R@5': r5,
            'MRR': mrr_val,
            'NDCG@10': ndcg10,
            'MAP': ap_score,
            'latency_ms': lat
        })
    elapsed_ms = (time.time() - t0) * 1000 / len(test_queries)
    speed_results[model_name] = round(elapsed_ms, 1)

df_eval = pd.DataFrame(eval_records)
print("✓ Evaluation completed!\n")

## 7. Tổng hợp và Hiển thị Kết quả

In [ ]:
summary = df_eval.groupby('model')[['P@5', 'R@5', 'MRR', 'NDCG@10', 'MAP']].mean().round(4).reindex(list(SEARCH_FNS.keys()))
summary['Latency (ms)'] = [speed_results[m] for m in summary.index]

print("=== FINAL EVALUATION RESULTS (LLM-AS-A-JUDGE GROUND TRUTH) ===")
print(summary)
print("===========================================================")

## 8. Vẽ Biểu đồ so sánh Độ chính xác (NDCG@10 & MAP & MRR)

In [ ]:
x = np.arange(len(summary))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, summary['NDCG@10'], width, label='NDCG@10', color=[MODEL_COLORS[m] for m in summary.index], alpha=0.9)
ax.bar(x, summary['MAP'], width, label='MAP', color=[MODEL_COLORS[m] for m in summary.index], alpha=0.6)
ax.bar(x + width, summary['MRR'], width, label='MRR', color=[MODEL_COLORS[m] for m in summary.index], alpha=0.3, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(summary.index)
ax.set_title('Retrieval Model Comparison (NDCG@10, MAP & MRR)', fontweight='bold')
ax.set_ylabel('Score')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(FIGURE_PATH / 'eval_precision_bar.png')
plt.show()
print(f"✓ Saved: {FIGURE_PATH / 'eval_precision_bar.png'}")

## 9. Vẽ Biểu đồ so sánh Độ trễ vs MAP

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for m in summary.index:
    ax.scatter(summary.loc[m, 'Latency (ms)'], summary.loc[m, 'MAP'], s=200, color=MODEL_COLORS[m], label=m)
    ax.annotate(
        m,
        xy=(summary.loc[m, 'Latency (ms)'], summary.loc[m, 'MAP']),
        xytext=(8, 4), textcoords='offset points',
        fontsize=9, fontweight='bold', color=MODEL_COLORS[m]
    )

ax.set_xscale('log')
ax.set_xlabel('Latency (ms) - Log Scale')
ax.set_ylabel('MAP')
ax.set_title('Quality (MAP) vs Latency (ms)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_PATH / 'eval_latency_quality.png')
plt.show()
print(f"✓ Saved: {FIGURE_PATH / 'eval_latency_quality.png'}")

## 10. Lưu Báo cáo JSON

In [ ]:
final_report = {
    'results': summary.to_dict(orient='index'),
    'per_query': df_eval.to_dict(orient='records')
}
with open(REPORT_PATH / 'evaluation_final.json', 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2)
print(f"✓ Saved final report JSON to: {REPORT_PATH / 'evaluation_final.json'}")